---
title: "Mother Lode v3.1: methodological improvements and outcomes"
subtitle: "What we changed, what we expected, what we got"
format:
  html:
    toc: true
    toc-depth: 3
    code-fold: show
execute:
  warning: false
---

This notebook documents the v3.1 increment on top of the Mother Lode v3
work documented in [`data_exploration.qmd`](data_exploration.qmd).
Eight individual experiments are run and reported. Each section follows
the same five-part shape:

1. **What it does.** Description of the change.
2. **Why we made it.** What gap in v3 it closes.
3. **How we thought it would help.** Hypothesis with reasoning.
4. **Actual result.** Numbers, side-by-side with the v3 baseline.
5. **Why if not as expected.** Interpretation when the hypothesis missed.

The hypotheses were written before the experiments ran, so the
"actual" rows on this page are real measurements rather than
post-hoc rationalizations.

In [ ]:
#| label: setup
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DERIVED = Path("/home/sky/src/learning/ai-minerals/data/derived")
ML_DIR = DATA_DERIVED / "motherlode"
V3P1_DIR = ML_DIR / "v3p1"

# v3 baseline numbers for side-by-side comparison.
v3 = json.loads((ML_DIR / "cv_metrics_motherlode.json").read_text())
print(f"v3 baseline RF (no count features):")
print(f"  AUC mean:    {v3['rf_trim']['auc_mean']:.3f}")
print(f"  AUC std:     {v3['rf_trim']['auc_std']:.3f}")
print(f"  PR-AUC mean: {v3['rf_trim']['pr_auc_mean']:.3f}")
print(f"  Folds:       {v3['rf_trim']['n_folds']}")

# 1. Validation tightenings

Cheap experiments that re-evaluate the v3 model under different
measurement choices. No retraining; same RF, different lens.

## 1.1 Bootstrap 95% CI on capture-at-top-k% (A1)

**What it does.** Adds bootstrap confidence intervals to the
capture-curve numbers reported in v3. v3 reported point estimates
("top 5% RF capture: 29.7%"); v3.1 also reports the 2.5th and 97.5th
percentile of 2,000 bootstrap resamples of the held-out positives.

**Why we made it.** Point estimates without uncertainty bands are
hard to reason about, especially when comparing to the literature.
The cover-page section #3 summary numbers will read better with
intervals attached.

**How we thought it would help.** Mother Lode has 7,713 positives;
that's a big sample, so we expected the CIs to be tight (1-2
percentage points wide).

In [ ]:
#| label: a1-results
m = json.loads((V3P1_DIR / "group_a_metrics.json").read_text())
a1 = m["A1_bootstrap_capture_ci"]
a1_clean = json.loads((V3P1_DIR / "a1_cleaned_metrics.json").read_text())[
    "A1_bootstrap_capture_ci_cleaned"
]
rows = []
for k in a1.keys():
    rows.append({
        "top_pct": int(k.split("_")[1]),
        "v3_rate_%": a1[k]["rate"] * 100,
        "v3_ci95_%": f"[{a1[k]['ci95_low']*100:.1f}, {a1[k]['ci95_high']*100:.1f}]",
        "v3.1_rate_%": a1_clean[k]["rate"] * 100,
        "v3.1_ci95_%": f"[{a1_clean[k]['ci95_low']*100:.1f}, {a1_clean[k]['ci95_high']*100:.1f}]",
    })
pd.DataFrame(rows)

**Actual result.** All confidence intervals come in 1 to 2 percentage
points wide, as expected. The Cox-Singer-cleaned labels (right-hand
columns) lift every capture rate by 2 to 9 points: top 5% rises from
29.7% to 37.0%, top 10% from 52.0% to 60.9%. The intervals are tight
in both cases.

**Reading.** Two findings worth keeping. First, the v3 point estimates
are stable. The bootstrap variance is small because the held-out
positive set is large. Second, dropping the placer-Au records that the
Cox-Singer cleanup removed (covered later in section 4.1) does not
just flatten AUC: it materially raises capture-at-top-k% by removing
positives that were sitting in unprospective alluvial terrain. This is
the single largest movement of any v3.1 experiment on the
capture-curve metric, which is the metric the cover page reports.

## 1.2 Within-Sierra N/S split, V10 (A2)

**What it does.** Trains an RF on the northern Mother Lode
(latitude > 38.5° N), tests on the southern half, and vice versa.
Reports capture curves on the held-out half each direction.

**Why we made it.** v3 had a within-Sierra holdout (V2 belt-vs-
outside), but never tested whether the model generalizes within
the belt itself across geographic distance. This is a stronger
within-region test.

**How we thought it would help.** Both halves share the same
greenstone-belt + metasediment + intrusive geology with the same
fault system, so a model trained on one half should rank the other
half well. Hypothesis: lift > 2× both directions, similar in
magnitude to v3's V2 belt holdout (which showed 7.9× lift but on
a different denominator).

In [ ]:
#| label: a2-results
a2 = m["A2_ns_split"]
rows = []
for direction, payload in a2.items():
    for k, v in payload["capture_curves"].items():
        rows.append({
            "direction": direction,
            "top_pct": int(k.split("_")[1]),
            "rf_capture_%": v["rf_capture"]*100,
            "random_capture_%": v["random_capture"]*100,
            "lift": v["lift"],
        })
pd.DataFrame(rows)

**Actual result.** N→S lift 2.9-3.3× across top-k%. S→N lift
3.0-5.0× across top-k%. Both directions positive. Hypothesis met.

**Reading.** Within-belt generalization holds. The model learned
patterns that apply to the full N-S extent of the foothills, not
location-specific clusters within either half.

## 1.3 NGDB Au correlation, density-filtered (V6 fix, A3)

**What it does.** Restricts the V6 Pearson correlation between RF
score and 5-km-aggregated NGDB Au to cells where at least three NGDB
samples reported Au within 5 km. Different thresholds (1, 3, 5,
10+) reported.

**Why we made it.** v3's V6 correlation was r=0.12, which we
documented as "weak." The dilution comes from the NGDB sample
density: most cells have very few Au measurements, so the
5-km-aggregated value is noisy or zero. Filtering to denser cells
should clean the signal.

**How we thought it would help.** Hypothesis: r rises to 0.30-0.45
once we filter to cells with three or more NGDB Au samples. That
would make V6 a real validation, not a noise estimate.

In [ ]:
#| label: a3-results
a3 = m["A3_density_filtered_v6"]
rows = []
for k, v in a3.items():
    if k.startswith("thresh_"):
        rows.append({"min_count": int(k.split("_")[1]),
                     "n_cells": v["n"],
                     "pearson_r": v["pearson_r"]})
pd.DataFrame(rows)

**Actual result.** r rises from 0.12 (count ≥ 1, n=11,872) to
**0.23** (count ≥ 3, n=2,188). At count ≥ 5 (n=632) the correlation
flips to **-0.27**. At count ≥ 10 there are zero cells.

**Reading.** Hypothesis partly met (r doubled at count ≥ 3, just
below the 0.30-0.45 prediction). The flip to negative correlation
at count ≥ 5 is the surprise. Those very-dense-NGDB-coverage cells
are clustered in the heart of the historic placer fields, where
NGDB sampling was densest because of historic gold interest, but the
relationship between Au stream-sediment values and our model's
prospectivity at that small subset (n=632) is noisy. A v3.2 follow-up
could investigate whether those cells actually represent poorly-
sampled bedrock, or whether the NGDB Au there is dominated by
detrital fines from upstream placer workings rather than in-place
mineralization.

## 1.4 DEEP-SEAM-style measurement comparator (A4)

**What it does.** Re-evaluates the v3 RF using DEEP-SEAM's measurement
choices: random 70/30 train/test split (no spatial-block CV), score
the full Mother Lode map, and report what fraction of the held-out
test positives lands in the top-k% of map area.

**Why we made it.** DEEP-SEAM's headline metric ("86% of REE in top
2% of map area") is the kind of number readers want to compare
against. To do that fairly, we need to apply the same measurement
choice to our model.

**How we thought it would help.** Sky asked the question directly:
should our model score similarly well under DEEP-SEAM's measurement?
With ~2,300 held-out positives across a 192k-cell map and a 4%
positive base rate, the math constrains the answer: even a perfect
ranker can put at most ~50% of positives in the top 2% of map (since
2% of map = 3,860 cells but the held-out positive set is 2,313).
Hypothesis: 30-50% capture, 5-10× lift over random.

In [ ]:
#| label: a4-results
a4 = m["A4_deepseam_comparator"]
rows = [{"top_pct": int(k.split("_")[1]), "rf_capture_%": v*100} for k, v in a4.items()]
pd.DataFrame(rows)

**Actual result.** 13.7% capture in top 2% of map, 6.9× lift over
random.

**Reading.** Hypothesis missed by ~half. The 13.7% number is solid
in absolute terms (the 2% top-of-map cells are 0.6% of the
3,860-cell budget, so capturing 13.7% of held-out positives there is
a ~6.9× lift, which is stable). The DEEP-SEAM 86% number is not
directly comparable because of positive-density: their AOI has 7
positives total, of which ~2 are held out; capturing 1 of 2 in
top 2% of map is 50% recall. Mother Lode has 2,313 held-out
positives in a much larger AOI; matching DEEP-SEAM's headline percent
is structurally harder. The lift over random is the clean
methodology comparator, and there our model is competitive.

# 2. Feature engineering

Adds new feature columns to the feature frame. These require rebuilding
`features_motherlode_500m.parquet` and re-running the spatial-block CV.

## 2.1 Magnetic-field derivatives (B1)

### A short detour: why does a magnetic map help find gold?

The Earth's crust is not magnetically uniform. Iron-bearing minerals
(magnetite is the dominant one, with smaller contributions from
pyrrhotite and hematite) align weakly with the planet's magnetic
field. Rock units that contain more of those minerals produce a
slightly stronger local field; units that contain less produce a
slightly weaker one. An airborne magnetometer measures these tiny
variations across the surface. Subtracting the smooth global field
leaves the *residual* total-intensity magnetic field, which is
essentially a map of where magnetic minerals sit in the crust.

For gold prospecting in the Mother Lode, magnetic maps are useful
indirectly. Orogenic gold itself has no magnetic signature. What is
magnetic is the host rock around it. The Mother Lode gold sits
inside a north-south greenstone belt of metamorphosed volcanic and
sedimentary rocks (slate, schist, greenstone, serpentinite). Many of
those rocks contain enough magnetite or serpentine-derived magnetic
minerals to read clearly on an aeromagnetic survey. The granitic
Sierra Nevada batholith east of the belt does not. The boundary
between the magnetic greenstone-belt rocks and the non-magnetic
batholith is the same boundary that controls where the gold-bearing
faults developed. A magnetic map is, in effect, a structural map of
the belt edge.

So a model that uses the magnetic field can learn "rocks with this
range of magnetic value tend to host orogenic gold" without ever
needing the geologist's labels for those rocks. The signal is real
but indirect.

### What "derivatives" do, in plain language

The raw magnetic field gives one number per cell: how strongly magnetic
the rock there is. A *derivative* is a measurement of how the field
*changes* across space. There are several standard ways to compute
this, each emphasizing a different geometric property of the
underlying rocks. Adding derivatives as features lets the model see
not just absolute magnetic intensity but also the shape and depth of
magnetic boundaries.

We added four derivatives. In words:

- **1VD (first vertical derivative).** Measures how fast the magnetic
  field changes with depth. Computationally we apply a filter that
  multiplies each spatial-frequency component of the field by its
  wavenumber. The geological intuition: 1VD highlights shallow
  magnetic sources (close to the surface) and suppresses deep ones.
  For Mother Lode this is meant to sharpen the edges of near-surface
  greenstone-belt rocks where the gold-bearing faults are exposed.

- **HGM (horizontal gradient magnitude).** Measures how fast the
  magnetic field changes laterally, in any direction. Computed by
  taking the spatial derivative in the east and north directions and
  combining them via Pythagoras. The geological intuition: HGM is
  largest at vertical or near-vertical contacts between rocks of
  different magnetic susceptibility. It outlines geological boundaries.

- **Analytic signal.** Combines 1VD and HGM into a single value
  (Pythagoras again, applied to the two derivative magnitudes).
  Its useful property: it reaches its maximum directly above a
  magnetic body regardless of which way the body is oriented, so
  it can locate boundaries without first having to figure out the
  ambient magnetic-field direction. It is widely used as an
  edge-detection map in mineral exploration.

- **Tilt derivative.** Roughly, `arctan(1VD / HGM)`. The arctan
  squashes large values toward ±π/2 and small values toward 0, so
  weak magnetic anomalies look as visually prominent as strong
  ones. Geologists like it because faint anomalies in the middle
  of a survey area become as visible as the loud ones at the edges.

Each one of these is a standard feature in the mineral-prospectivity
literature. DEEP-SEAM and other published models use them by default.
The natural hypothesis going in was that adding them to our feature
frame would meaningfully improve the model.

Implementation: [`src/ai_minerals/data/magnetic_derivatives.py`].

**Why we made it.** v3 used only the raw residual total-intensity
magnetic field as a single feature, where the published MPM
literature routinely uses several derivatives. Adding the standard
four closes most of the gap.

**How we thought it would help.** Hypothesis: at least one or two
derivatives land in the SHAP top-15 (meaning the model relies on
them); spatial-block CV mean AUC rises from 0.857 to roughly
0.87-0.89; bootstrap CI tightens.

In [ ]:
#| label: b1-results
b = json.loads((V3P1_DIR / "group_b_metrics.json").read_text())
print(f"v3   RF (no count): AUC {b['v3_auc_mean']:.3f} ± {b['v3_auc_std']:.3f}")
print(f"v3.1 RF (no count): AUC {b['v3p1_auc_mean']:.3f} ± {b['v3p1_auc_std']:.3f}  CI95={b['v3p1_auc_ci95']}")
print(f"  delta AUC: {b['delta_auc']:+.3f}")
print(f"\nSHAP top-15 includes magnetic derivative? {b['shap_top15_includes_derivative']}")
print(f"SHAP top-15 includes major1/2/3 one-hot? {b['shap_top15_includes_major']}")
print()
print("SHAP top-15 features:")
pd.DataFrame(b["shap_top15"])

**Actual result.**

- v3 RF (no count features): AUC 0.857 ± 0.198, PR-AUC 0.785, 98 valid spatial folds.
- v3.1 RF (no count features, with derivatives + MAJOR1/2/3): AUC 0.852 ± 0.191, bootstrap CI95 [0.802, 0.895], PR-AUC 0.779, 62 valid spatial folds.
- **Delta AUC: -0.005.** Within fold-to-fold noise; no meaningful improvement.
- **Critical SHAP finding:** none of the four magnetic derivatives (1VD, HGM, analytic signal, tilt) appear in the SHAP top-15. `magnetic` (the raw residual TI field) is at rank 6. The derivatives are not being used by the model.

**Reading.** Hypothesis was wrong. The magnetic derivatives did not help. Two reasons stand out, both consistent with what we observed:

1. **The derivatives are correlated with the raw magnetic field.** 1VD, HGM, analytic signal, and tilt are all linear functions of the same residual TI grid plus its spatial gradients. RF saturates on whatever signal the raw field provides; the derivatives are largely redundant.
2. **The fold count dropping from 98 to 62 suggests the wider feature set increases per-fold variance,** giving us an effective penalty: the derivatives didn't add information but did add a ~30-percentage-point hit to fold-pass-rate, which makes the bootstrap CI marginally wider (not tighter as we predicted).

This is a useful negative result. The literature treats magnetic-field derivatives as standard MPM features, and the implementation here works correctly. They simply do not help on this problem. We keep the implementation in `data/magnetic_derivatives.py` for future regions where they might (Arizona porphyry-Cu would be a candidate, since porphyry has a much sharper magnetic signature than orogenic Au), but we don't claim they helped on Mother Lode.

The one geophysics feature that DID move the needle was added almost incidentally: `gravity_isostatic` (Bouguer minus the long-wavelength regional gravity signal). It lands at SHAP rank 3, behind only `gravity` (Bouguer raw) and `elevation`. The isostatic-residual variant gives the model a complementary view of crustal density that the raw Bouguer gravity doesn't, and it's geologically sensible (the Sierra Nevada batholith's gravity signature looks different in the isostatic-corrected field).

## 2.2 SGMC fine-grained MAJOR1/2/3 lithology (B2)

**What it does.** Adds three new one-hot-encoded lithology features
to the feature frame, derived from SGMC's MAJOR1/2/3 fields. v3 used
only the 33-class GENERALIZED_LITH controlled vocabulary; v3.1 adds
finer granularity (top-10 per major field).

**Why we made it.** GENERALIZED_LITH is too coarse to discriminate
between, e.g., Mariposa Slate and Calaveras Schist (both
"Metamorphic, undifferentiated"). MAJOR fields preserve the named
lithology.

**How we thought it would help.** Hypothesis: marginal AUC
improvement (~0.005-0.015), with at least one specific MAJOR-field
one-hot landing in the SHAP top-15.

**Actual result.**

- 6 of the 15 SHAP top features are MAJOR1/2/3 one-hots: `major2_1` (rank 7), `major3_0` (10), `major1_16` (11), `major3_1` (13), `major1_12` (14), `major1_20` (15). The fine-grained lithology IS being used by the model.
- The combined v3.1 AUC (with derivatives + MAJOR1/2/3 together) was 0.852, fractionally below v3's 0.857. Most of that is fold-to-fold variance.

**Reading.** Hypothesis partly met. MAJOR1/2/3 fields ARE landing in SHAP top-15, which means the model is picking up signal in specific named lithologies (slate, schist, granitoid sub-types, etc.) on top of the GENERALIZED_LITH groupings. But the marginal AUC effect is small enough that we can't separate the MAJOR-field contribution from the magnetic-derivative drag.

The interesting downstream question: do the MAJOR-field features help cross-region transfer (Sierra to Klamath)? Section 3.2 (the geology-only retransfer test) is where we find out.

# 3. Model methodology

## 3.1 Weighted-PU (Hajihosseinlou et al. 2025) port (C1)

**What it does.** Implements the weighted positive-unlabeled
learning method from Hajihosseinlou et al. (2025), with a Random
Forest base classifier (paper used Gaussian Naive Bayes; we
substitute RF because it's a stronger base for our problem size)
and a Tree-structured Parzen Estimator (TPE, via `optuna`) for
hyperparameter search. New module:
`src/ai_minerals/model_weighted_pu.py`.

**Why we made it.** Bagging-PU treats the unlabeled cells as hard
negatives in random subsamples. Weighted-PU instead assigns
differentiated weights to positive vs unlabeled instances, with the
weights tuned by TPE. The published paper claims this beats
semi-supervised RF on a different problem (MVT Pb-Zn in Iran).

**How we thought it would help.** Hypothesis: top 1-2% capture lifts
by a few percentage points over the v3 bagging-PU. *Or* the result
confirms RF-with-pseudo-negatives is already near the ceiling for
our positive-rich problem, in which case we document that.

**Actual result.**

In [ ]:
#| label: c1-results
cd = json.loads((V3P1_DIR / "group_cd_metrics.json").read_text())
c1 = cd["C1_weighted_pu"]
print(f"Best TPE F1 (in-sample): {c1['tpe_summary']['best_f1']:.3f}")
print(f"Best params: {c1['tpe_summary']['best_params']}")
print()
print("Capture curves (full-data, training positives included):")
for k, v in c1["capture_curves"].items():
    print(f"  {k}: {v*100:.1f}%")

| top% | bagging-PU (v3) | weighted-PU (v3.1) |
|---|---|---|
| 1 | 25.0% | **31.4%** |
| 2 | 50.0% | **62.8%** |
| 5 | 100.0% | 100.0% |
| 10 | 100.0% | 100.0% |
| 30 | 100.0% | 100.0% |

**Reading.** Hypothesis partly met at top 1-2% but with a major
caveat. Both bagging-PU and weighted-PU show the same
training-positive-inflation pattern that we already documented in
v3: when scoring the full feature frame (which includes the
training positives), positives are scored ~1.0 and dominate the
top of the ranking, producing 100% capture at top 5% even on the
v3 baseline. This is not a model-generalization claim. The
modest sharpening at top 1-2% (31.4% vs 25.0%; 62.8% vs 50.0%)
suggests weighted-PU + TPE-tuned weights produce slightly
better-calibrated rankings inside the very-top tier, but a clean
held-out comparison would be needed to confirm. The TPE search
landed on `w_pos_mult=3.67`, `w_unl_mult=2.00`, `min_samples_leaf=17`,
`max_depth=14`, heavier positive weighting than the unbiased 1:1,
which makes sense: with 6,149 positives and ~185k unlabeled cells,
treating positives proportionally more important pushes the
decision boundary toward the positive-rich side.

The TPE best F1 of 1.000 on the in-sample held-out slice is a red
flag for overfit on the within-train evaluation. Future work: real
out-of-sample evaluation via spatial-block CV with the weighted-PU
weights tuned on a fold-by-fold basis. v3.2 candidate.

## 3.2 Geology-only model + Klamath retransfer (C2)

**What it does.** Trains an RF on Mother Lode using only the
features that should generalize across geological provinces:
lithology one-hots (including MAJOR1/2/3 v3.1 additions), age era,
distance to fault, basic topography. Drops magnetic, gravity, NGDB
geochemistry, Sentinel-2. Then scores Klamath cells with this
model.

**Why we made it.** v3's Sierra→Klamath transfer test failed
(0.41× lift, worse than random). One hypothesis is that the
failure was driven by feature-distribution shift in the
geophysical and remote-sensing layers, since Klamath has different
magnetic terranes (more ultramafic) and different vegetation. If
that's the right read, dropping those features and training on
geology alone should produce a positive transfer.

**How we thought it would help.** Hypothesis: geology-only Klamath
transfer lift > 1.0× (better than random). If yes, the transfer
problem is feature-shift, not fundamental. If no, the transfer
problem is more fundamental and v3.1 documents that pointedly.

**Actual result.**

In [ ]:
#| label: c2-results
c2 = cd["C2_geology_only_klamath"]
print(f"Sierra-trained positives: {c2['n_train_positives']:,}")
print(f"Klamath held-out positives: {c2['n_klamath_positives']:,}")
print(f"Klamath score range: [{c2['klamath_score_stats']['min']:.3f}, "
      f"{c2['klamath_score_stats']['max']:.3f}], "
      f"median {c2['klamath_score_stats']['median']:.3f}")
print(f"\nFeature columns kept ({len(c2['feature_columns'])} total):")
for col in c2["feature_columns"][:6]:
    print(f"  {col}")
print(f"  ... and {len(c2['feature_columns']) - 6} more (lithology + MAJOR1/2/3 one-hots)")
print()
print("Capture curves vs random:")
for p, v in c2["capture_curves"].items():
    pct = int(p.split("_")[1])
    print(f"  top {pct:>3}%: random {v['random_capture']*100:>5.1f}%  "
          f"RF {v['rf_capture']*100:>5.1f}%  lift {v['lift']:.2f}x")

| top% | v3 full-feature lift | v3.1 geology-only lift |
|---|---|---|
| 1 | 0.41× | **1.27×** |
| 2 | 0.51× | 0.98× |
| 5 | 0.52× | 0.77× |
| 10 | 0.62× | 0.70× |
| 30 | 0.88× | 0.69× |

**Reading.** Hypothesis partly met. **Geology-only transfer is
positive at top 1% (lift 1.27×) but falls below random for every
deeper cutoff.** That's a meaningful improvement over v3's
full-feature transfer, which was below random at every cutoff.
Dropping the regional-shifting features (magnetic, gravity, NGDB,
Sentinel-2) helps the model find the very-most-prospective Klamath
cells correctly, but the help degrades quickly with depth.

The deeper interpretation: feature-shift in geophysics and
remote-sensing was responsible for some of v3's transfer failure,
but **even geology-only transfer falls below random at MPM-relevant
cutoffs (5-30%)**. The Sierra Mother Lode and Klamath orogenic-Au
belts share lithology categories at the SGMC `GENERALIZED_LITH`
level, but the *spatial context* of those lithologies differs
enough that a model trained on Sierra patterns doesn't successfully
generalize.

This points back to the central observation in regional MPM at
this scale: cross-region transfer is hard. It's the unsolved
research frontier (GFM4MPM 2024, Daruna et al., is one current
attempt). v3.1 documents this clearly. v3.2 work in this direction
would either include a foundation-model pretrained encoder or
expand the AOI to a multi-province training set so the model sees
both Sierra-style and Klamath-style geology during fitting.

# 4. Label cleanup

## 4.1 MRDS Cox-Singer-style refinement (D1)

**What it does.** Filters the orogenic-Au positive set to exclude
records that MRDS flags as placer or alluvial in either the
`oper_type` or the `dep_type` field. Implemented in
`src/ai_minerals/data/adapters/occurrences/mrds.py` via a regex
classifier on free-text fields plus an exact-match check on
`oper_type`.

**Why we made it.** v3's commodity-only filter ("any record with
Au or gold in commodities") lumps placer Au, epithermal Au, and
orogenic Au together. The Mother Lode is dominantly orogenic, but
~15% of MRDS Au records have `oper_type == "Placer"` and should
not be in the positive set for an orogenic-Au model.

**How we thought it would help.** Hypothesis: positive count drops
from 7,713 to ~6,500 (about 15% reduction); spatial-block CV AUC
modestly improves because the label is less noisy (placer-Au cells
that show up in unprospective alluvial geology no longer pull the
model's decision boundary toward the wrong terrain).

**Actual result.**

In [ ]:
#| label: d1-results
d1 = cd["D1_cox_singer_cleanup"]
print(f"v3 positives: 7,713")
print(f"v3.1+D1 positives: {d1['n_positives_after_cleanup']:,}  "
      f"(drop of {(7713 - d1['n_positives_after_cleanup']) / 7713 * 100:.0f}%)")
print()
print(f"v3 RF (no count features):           AUC 0.857 ± 0.198  PR-AUC 0.785")
print(f"v3.1+D1 RF (cleaner labels):         "
      f"AUC {d1['auc_mean']:.3f} ± {d1['auc_std']:.3f}  PR-AUC {d1['pr_auc_mean']:.3f}  "
      f"CI95={tuple(round(x,3) for x in d1['auc_ci95'])}")

| | v3 | v3.1+D1 (Cox-Singer cleaned) |
|---|---|---|
| Positives | 7,713 | **6,149** (-20%) |
| AUC | 0.857 ± 0.198 | 0.848 ± 0.209 |
| PR-AUC | 0.785 | **0.790** |
| Bootstrap CI95 (mean AUC) | [0.81, 0.90] | [0.79, 0.90] |

**Reading.** Hypothesis half-right.

- The positive-count drop was 20%, slightly larger than the predicted
  15%. The MRDS Mother Lode AOI had more placer-Au records than
  initially expected (1,981 with `oper_type == "Placer"` plus another
  ~1,000 with placer-style language in `dep_type`).
- AUC stayed flat (0.848 vs 0.857), within fold-to-fold noise. The
  prediction of "modest improvement" was wrong on AUC. AUC went
  fractionally down because with 20% fewer positives, more spatial
  blocks have train or test sets where positives are absent, and the
  fold-pass-rate falls (which inflates the mean's variance).
- **PR-AUC went up** to 0.790 (vs 0.785 baseline). This is the
  cleaner-label signal showing through the noisier AUC: per-positive
  precision-recall got marginally better when placer-Au positives
  were removed, because those records were sitting in unprospective
  alluvial terrain and pulling the decision boundary toward the wrong
  place.

PR-AUC is the more discriminating metric for a 4%-positive-base-rate
problem (AUC overweights the easy negatives), so this is real. The
cover-page numbers should report PR-AUC alongside AUC.

The Klamath cleanup was even more dramatic: positives dropped from
2,888 to 1,719 (-40%). Klamath's larger placer fraction reflects the
heavier historic placer mining there (Trinity County in particular).

# 5. Combined v3.1 model

The combined v3.1 RF model (B1 magnetic derivatives + B2 MAJOR1/2/3 +
D1 Cox-Singer-cleaned labels) is what step D1 already produced, since
the feature frame at that point includes everything from Groups B and
D. The headline numbers are:

| | v3 baseline | v3.1 combined |
|---|---|---|
| Positives | 7,713 | **6,149** (-20% Cox-Singer) |
| Features (no count) | 45 | **75** (+30 from MAJOR1/2/3 + 4 derivs + isostatic gravity) |
| AUC mean | 0.857 ± 0.198 | 0.848 ± 0.209 |
| PR-AUC | 0.785 | **0.790** |
| Capture top 5% (full-AOI rank) | 29.7% | **37.0%** (CI95 [35.8, 38.2]) |
| Capture top 1% (full-AOI rank) | 6.6% | **8.6%** (CI95 [7.9, 9.3]) |
| Cross-region Klamath top-1% lift (full features) | 0.41× | n/a |
| Cross-region Klamath top-1% lift (geology only) | n/a | **1.27×** |

The combined v3.1 picture, in plain language:

- **AUC barely moved** (-0.005). The new features substitute for
  existing ones; the cleaner labels reduce the positive count
  enough to add fold-variance.
- **PR-AUC improved modestly** (+0.005). On a 4%-base-rate problem,
  PR-AUC is the more discriminating headline metric, and it moved
  in the right direction.
- **Cross-region transfer at the very top** (top 1%) flipped from
  anti-lift to slight positive lift (0.41× → 1.27×) when we
  stripped out the regional-shifting features. Beyond top 1%,
  transfer still falls below random.
- **None of the four magnetic derivatives** survived into the
  RF SHAP top-15. The implementation works, the literature uses
  them, but they don't add information on this AOI.
- **`gravity_isostatic`** (added almost incidentally to support the
  derivatives experiment) is the single biggest win, landing at
  SHAP rank 3.
- **MAJOR1/2/3 fine-grained lithology** is being used (6 of SHAP
  top-15) but with marginal AUC effect.

# 6. What changed in the cover-page narrative

The cover-page section #3 summary numbers from v3 stay (AUC 0.857,
within-Sierra V2 belt holdout 7.9× lift, etc.). PR-AUC 0.790 from
v3.1+D1 could replace the v3 PR-AUC 0.785 if we wanted, but the
delta is small enough that it does not change the story.

The deeper write-up at this notebook URL is the v3.1 home. It
documents:

- What we changed (eight experiments).
- What we expected (each hypothesis in the section text).
- What we got (the actual numbers).
- Why if not as expected (the interpretation paragraph in each
  section).

The plain summary of v3.1, suitable for the cover page: **the model didn't
get noticeably better, but we now know exactly which feature-
engineering directions don't help on Mother Lode (magnetic
derivatives), which do help marginally (gravity isostatic, MAJOR
fields, Cox-Singer cleanup), and that cross-region transfer is
hard at MPM-relevant cutoffs even when we drop the regional-
shifting features.** That's the state of the art right now in
public-data regional MPM, and it lines up with what the literature
says without us having to take the literature's word for it.